# Consolidation Pipeline

In [11]:
import pandas as pd
import numpy as np

us = pd.read_csv("us_region_sales-1.csv")
apac = pd.read_csv("apac_region_sales-1.csv")
eu = pd.read_csv("eu_region_sales.csv")
rates = pd.read_csv("exchange_rates.csv")
product_master = pd.read_csv("product_master.csv")
bank = pd.read_csv("bank_statement.csv")
gl = pd.read_csv("general_ledger_cash.csv")

In [12]:
# Check if everything got imported correctly

print("US Sales:", us.shape)

print("APAC Sales:", apac.shape)

print("EU Sales:", eu.shape)

print("Exchange Rates:", rates.shape)

print("Product Master:", product_master.shape)

print("Bank Statement:", bank.shape)

print("General Ledger:", gl.shape)

US Sales: (715, 6)
APAC Sales: (510, 8)
EU Sales: (615, 6)
Exchange Rates: (1085, 3)
Product Master: (10, 3)
Bank Statement: (61, 4)
General Ledger: (60, 5)


In [13]:
# Inspect the sales files

display(us.head())

display(apac.head())

display(eu.head())

,Order_No,Sale_Date,Store_City,Product_Name,Qty_Sold,Unit_Price_USD
0,US-100001,01/01/2026,Denver,Drawer Slide 22in,13,18.9
1,US-100002,01/01/2026,Denver,Drawer Slide 22in,13,18.9
2,US-100005,01/01/2026,Dallas,Soft-Close Hinge,2,12.3
3,US-100006,01/02/2026,Denver,Adjustable Shelf Pin Set,12,6.4
4,US-100009,01/02/2026,Denver,Soft-Close Hinge,7,12.3


,txn_ref,txn_date_ddmmyyyy,city,product,qty,unit_price_local,currency,tax_pct
0,AP-300001,01-01-2026,Melbourne,Corner Bracket 3in,8,4.74,AUD,10.0
1,AP-300004,01-01-2026,Singapore,Cabinet Pull - Brushed Nickel,4,6.78,SGD,9.0
2,AP-300006,02-01-2026,Singapore,Locking Cam Latch,7,5.45,SGD,9.0
3,AP-300009,03-01-2026,Singapore,Soft-Close Hinge,10,18.12,SGD,9.0
4,AP-300011,04-01-2026,Sydney,Soft-Close Hinge,8,20.82,AUD,10.0


,order_ref,date,city,item,line_total_eur_incl_vat,units
0,EU-200001,2026-01-01,Berlin,DRAWER SLIDE 22IN,189.00,9
1,EU-200004,2026-01-01,Lyon,CORNER BRACKET 3IN,9.33,3
2,EU-200007,2026-01-03,Milan,LOCKING CAM LATCH,201.39,49
3,EU-200009,2026-01-03,Milan,LOCKING CAM LATCH,53.43,13
4,EU-200012,2026-01-04,Berlin,SOFT-CLOSE HINGE,191.38,14


# Original Revenue Calculations

In [16]:
# US Original Revenue

us["Original_Revenue"] = (
    us["Qty_Sold"] * us["Unit_Price_USD"]
)

# Currency
us["currency"] = "USD"

# Check calculations

display(
    us[[
        "Order_No",
        "Qty_Sold",
        "Unit_Price_USD",
        "Original_Revenue",
        "currency"
    ]].head(10)
)

,Order_No,Qty_Sold,Unit_Price_USD,Original_Revenue,currency
0,US-100001,13,18.9,245.7,USD
1,US-100002,13,18.9,245.7,USD
2,US-100005,2,12.3,24.6,USD
3,US-100006,12,6.4,76.8,USD
4,US-100009,7,12.3,86.1,USD
5,US-100012,6,2.8,16.8,USD
6,US-100013,5,2.8,14.0,USD
7,US-100014,14,12.3,172.2,USD
8,US-100015,65,9.8,637.0,USD
9,US-100017,5,2.8,14.0,USD


In [20]:
# APAC Original Revenue

apac["Original_Revenue"] = (
    apac["qty"] * apac["unit_price_local"]
)


# Check calculations

display(
    apac[[
        "txn_ref",
        "qty",
        "unit_price_local",
        "Original_Revenue",
        "currency"
    ]].head(10)
)

,txn_ref,qty,unit_price_local,Original_Revenue,currency
0,AP-300001,8,4.74,37.92,AUD
1,AP-300004,4,6.78,27.12,SGD
2,AP-300006,7,5.45,38.15,SGD
3,AP-300009,10,18.12,181.20,SGD
4,AP-300011,8,20.82,166.56,AUD
5,AP-300012,5,1051.00,5255.00,JPY
6,AP-300013,13,4.74,61.62,AUD
7,AP-300014,5,12.86,64.30,AUD
8,AP-300015,16,804.00,12864.00,JPY
9,AP-300016,2,1609.00,3218.00,JPY


In [26]:
# EU Original Revenue

eu["Original_Revenue"] = (
    eu["line_total_eur_incl_vat"]
)

eu["currency"] = "EUR"

# Check Calculations

display(
    eu[[
        "order_ref",
        "units",
        "line_total_eur_incl_vat",
        "Original_Revenue",
        "currency"
    ]].head(10)
)

,order_ref,units,line_total_eur_incl_vat,Original_Revenue,currency
0,EU-200001,9,189.00,189.00,EUR
1,EU-200004,3,9.33,9.33,EUR
2,EU-200007,49,201.39,201.39,EUR
3,EU-200009,13,53.43,53.43,EUR
4,EU-200012,14,191.38,191.38,EUR
5,EU-200014,7,28.77,28.77,EUR
6,EU-200016,9,123.03,123.03,EUR
7,EU-200017,12,85.32,85.32,EUR
8,EU-200020,8,24.88,24.88,EUR
9,EU-200022,3,16.32,16.32,EUR


# Date Parsing

In [37]:
# Parse US dates to mm/dd/yyyy

us["Parsed_Date"] = pd.to_datetime(
    us["Sale_Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

display(
    us[["Order_No", "Sale_Date", "Parsed_Date"]].head(10)
)

,Order_No,Sale_Date,Parsed_Date
0,US-100001,01/01/2026,2026-01-01
1,US-100002,01/01/2026,2026-01-01
2,US-100005,01/01/2026,2026-01-01
3,US-100006,01/02/2026,2026-01-02
4,US-100009,01/02/2026,2026-01-02
5,US-100012,01/03/2026,2026-01-03
6,US-100013,01/03/2026,2026-01-03
7,US-100014,01/03/2026,2026-01-03
8,US-100015,01/03/2026,2026-01-03
9,US-100017,01/04/2026,2026-01-04


In [34]:
# Parse APAC dates to yyyy-mm-dd

apac["Parsed_Date"] = pd.to_datetime(
    apac["txn_date_ddmmyyyy"],
    format="%d-%m-%Y",
    errors="coerce"
)

display(
    apac[[
        "txn_ref",
        "txn_date_ddmmyyyy",
        "Parsed_Date"
    ]].head(10)
)

,txn_ref,txn_date_ddmmyyyy,Parsed_Date
0,AP-300001,01-01-2026,2026-01-01
1,AP-300004,01-01-2026,2026-01-01
2,AP-300006,02-01-2026,2026-01-02
3,AP-300009,03-01-2026,2026-01-03
4,AP-300011,04-01-2026,2026-01-04
5,AP-300012,04-01-2026,2026-01-04
6,AP-300013,04-01-2026,2026-01-04
7,AP-300014,05-01-2026,2026-01-05
8,AP-300015,05-01-2026,2026-01-05
9,AP-300016,05-01-2026,2026-01-05


In [36]:
# Parse EU dates to yyyy-mm-dd

eu["Parsed_Date"] = pd.to_datetime(
    eu["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

display(
    eu[[
        "order_ref",
        "date",
        "Parsed_Date"
    ]].head(10)
)

,order_ref,date,Parsed_Date
0,EU-200001,2026-01-01,2026-01-01
1,EU-200004,2026-01-01,2026-01-01
2,EU-200007,2026-01-03,2026-01-03
3,EU-200009,2026-01-03,2026-01-03
4,EU-200012,2026-01-04,2026-01-04
5,EU-200014,2026-01-04,2026-01-04
6,EU-200016,2026-01-04,2026-01-04
7,EU-200017,2025-01-05,2025-01-05
8,EU-200020,2026-01-05,2026-01-05
9,EU-200022,2026-01-05,2026-01-05


# Identifying Invalid Dates